In [4]:
!pip install anthropic dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

In [7]:
load_dotenv()

True

In [8]:
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [10]:
message = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Hello, Claude",
        }
    ],
)

In [15]:
print(message.content[0].text)

Hello! How are you doing today? Is there anything I can help you with?


In [73]:
class Agent:
    def __init__(self,system=""):
        self.system = system
        self.messages = []

    def __call__(self,message):
        self.messages.append({"role":"user","content":message})
        result = self.execute()
        self.messages.append({"role":"assistant","content":result})
        return result

    def execute(self):
        if self.system:
            completion = client.messages.create(
                model="claude-opus-4-8",
                max_tokens=1024,
                messages=self.messages,
                system=self.system
            )
            return completion.content[0].text

In [74]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [75]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")


known_actions = {
    "calculate":calculate,
    "average_dog_weight": average_dog_weight
}

In [76]:
abot = Agent(prompt)

In [50]:
result = abot("How much does a toy poodle weigh?")

Thought: I should look up the toy poodle's weight using average_dog_weight

Action: average_dog_weight: Toy Poodle
PAUSE


In [51]:
result = average_dog_weight("Toy Poodle")

In [52]:
next_prompt = "Observation : {}".format(result)

In [53]:
result = abot(next_prompt)

Answer: A toy poodle weighs 7 lbs


In [54]:
abot.messages

[{'role': 'user', 'content': 'How much does a toy poodle weigh?'},
 {'role': 'assistant',
  'content': "Thought: I should look up the toy poodle's weight using average_dog_weight\n\nAction: average_dog_weight: Toy Poodle\nPAUSE"},
 {'role': 'user',
  'content': 'Observation : a toy poodles average weight is 7 lbs'},
 {'role': 'assistant', 'content': 'Answer: A toy poodle weighs 7 lbs'}]

In [77]:
abot = Agent(prompt)

In [56]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

In [57]:
abot(question)

Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together. Let me start with the Border Collie.

Action: average_dog_weight: Border Collie
PAUSE


In [58]:
result = average_dog_weight("Border Collie")

In [59]:
next_prompt = "Observation : {}".format(result)

In [60]:
abot(next_prompt)

Thought: Now I need to find the average weight of a Scottish Terrier.

Action: average_dog_weight: Scottish Terrier
PAUSE


In [61]:
result = average_dog_weight("Scottish Terrier")

In [62]:
next_prompt = "Observation : {}".format(result)

In [63]:
abot(next_prompt)

Thought: Now I can add the two weights together: 37 lbs + 20 lbs.

Action: calculate: 37 + 20
PAUSE


In [64]:
result = calculate("37 + 20")

In [65]:
next_prompt = "Observation : {}".format(result)

In [66]:
abot(next_prompt)

Answer: The combined weight of your Border Collie (37 lbs) and Scottish Terrier (20 lbs) is 57 lbs.


In [68]:
import re
action_re = re.compile(r'^Action: (\w+): (.*)$')


In [80]:
def query(question,max_retries=5):
    i = 0
    abot = Agent(prompt)
    next_prompt = question
    while i < max_retries:
        result = abot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            action,action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation {}".format(observation)
        else:
            return

In [81]:
query("I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight")

Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together. Let me start with the Border Collie.

Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: Now I need to find the average weight of a Scottish Terrier.

Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: Now I have both weights. The Border Collie is 37 lbs and the Scottish Terrier is 20 lbs. I need to add them together.

Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of your Border Collie (37 lbs) and Scottish Terrier (20 lbs) is 57 lbs.
